In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

In [0]:
df_e= spark.read.table("healthverity_claims_sample_patient_dataset.hv_claims_sample.enrollment")
df_c= spark.read.table("healthverity_claims_sample_patient_dataset.hv_claims_sample.pharmacy_claim")


In [0]:
df=df_e.filter(df_e["patient_state"]=="CA")

In [0]:
df.display()

patient_id,patient_gender,patient_year_of_birth,patient_zip3,patient_state,date_start,date_end,benefit_type,pay_type
013f04f686d1d807de93678c6bf38b88,M,1947,921,CA,2017-01-01,2024-02-29,MEDICAL,COMMERCIAL
013f04f686d1d807de93678c6bf38b88,M,1947,921,CA,2017-01-01,2024-02-29,PHARMACY,COMMERCIAL
03723da99f7802666e30e4ccfb011d13,M,2008,951,CA,2017-01-01,2024-02-29,MEDICAL,MEDICAID
03723da99f7802666e30e4ccfb011d13,M,2008,951,CA,2017-01-01,2024-02-29,PHARMACY,MEDICAID
09140ac77017bc3881db171cc10b4b0d,M,1934,945,CA,2018-01-01,2023-12-31,MEDICAL,MEDICARE
09140ac77017bc3881db171cc10b4b0d,M,1934,945,CA,2017-01-01,2017-12-31,PARTIAL MEDICAL,MEDICARE
09140ac77017bc3881db171cc10b4b0d,M,1934,945,CA,2018-01-01,2023-12-31,PHARMACY,MEDICARE
09a22359accc92df9319be625acd13d3,F,1968,920,CA,2019-01-01,2024-01-31,MEDICAL,COMMERCIAL
09a22359accc92df9319be625acd13d3,F,1968,920,CA,2019-01-01,2024-01-31,PHARMACY,COMMERCIAL
146166ee18d6bddd38c5e375df754431,F,1992,930,CA,2017-01-01,2024-02-29,MEDICAL,MEDICAID


In [0]:
df.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- PhotonResultStage
   +- PhotonScan parquet healthverity_claims_sample_patient_dataset.hv_claims_sample.enrollment[patient_id#13279,patient_gender#13280,patient_year_of_birth#13281,patient_zip3#13282,patient_state#13283,date_start#13284,date_end#13285,benefit_type#13286,pay_type#13287] DataFilters: [isnotnull(patient_state#13283), (patient_state#13283 = CA)], DictionaryFilters: [(patient_state#13283 = CA)], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://healthverity-warehouse-green-prod-581191604223-us-east-1/f72..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<patient_id:string,patient_gender:string,patient_year_of_birth:string,patient_zip3:string,p..., RequiredDataFilters: [isnotnull(patient_state#13283), (patient_state#13283 = CA)]


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  missing = enrollment
  partial = 
  full    =

In [0]:
import pyspark.sql.functions as F
df1 = df.groupBy("patient_state").agg(F.max("patient_year_of_birth"))

In [0]:
df1.display()

patient_state,max(patient_year_of_birth)
CA,2019


In [0]:
df1.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonSort [patient_state#13382 ASC NULLS FIRST]
         +- PhotonGroupingAgg(keys=[patient_state#13382], functions=[finalmerge_max(merge max#13393) AS max(patient_year_of_birth)#13388])
            +- PhotonGroupingAgg(keys=[patient_state#13382], functions=[partial_max(patient_year_of_birth#13380) AS max#13393])
               +- PhotonScan parquet healthverity_claims_sample_patient_dataset.hv_claims_sample.enrollment[patient_year_of_birth#13380,patient_state#13382] DataFilters: [isnotnull(patient_state#13382), (patient_state#13382 = CA)], DictionaryFilters: [(patient_state#13382 = CA)], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://healthverity-warehouse-green-prod-581191604223-us-east-1/f72..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<patient_year_of_birth:string,patient_state:string>, RequiredDataFilters: [isnotn